# 02 — Web-Scraping Plagiarism Check (no training)

This notebook demonstrates the **external plagiarism** path from the proposal:
given a document, find candidate web pages, scrape their text, and measure similarity.

**No training here** — we reuse the fine-tuned model from notebook 01 if it exists,
otherwise the stock pretrained `all-MiniLM-L6-v2`.

Pipeline: `keywords -> search -> scrape (BeautifulSoup) -> cosine similarity -> report`.

> Search engines block bots, so we use the DuckDuckGo HTML endpoint (no API key) with a
> graceful fallback to user-supplied URLs. For production, swap in a real search API.

In [ ]:
# 1. Install (run once)
# !pip install requests beautifulsoup4 sentence-transformers torch
import re
import requests
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer, util
import os

HEADERS = {'User-Agent': 'Mozilla/5.0 (plagiarism-detector-demo)'}

In [ ]:
# 2. Load the model (fine-tuned if available, else pretrained)
FINE_TUNED = '../backend/detector/models/plagiarism-sbert'
model_path = FINE_TUNED if os.path.isdir(FINE_TUNED) else 'all-MiniLM-L6-v2'
print('Loading model:', model_path)
model = SentenceTransformer(model_path)

In [ ]:
# 3. Keyword extraction (simple + explainable): top frequent non-stopword tokens
STOPWORDS = set('the a an and or but of to in on for is are was were be been being with as at by '
                'this that these those it its from we you they he she i our your their'.split())

def keywords(text, k=6):
    words = re.findall(r'[a-zA-Z]{4,}', text.lower())
    freq = {}
    for w in words:
        if w not in STOPWORDS:
            freq[w] = freq.get(w, 0) + 1
    return [w for w, _ in sorted(freq.items(), key=lambda x: -x[1])[:k]]

sample_doc = (
    'Photosynthesis is the process by which green plants convert sunlight into chemical energy. '
    'Chlorophyll in the leaves absorbs light and uses it to turn carbon dioxide and water into glucose.'
)
print('Keywords:', keywords(sample_doc))

In [ ]:
# 4. Search for candidate URLs (DuckDuckGo HTML, no API key)
def search_urls(query, max_results=5):
    try:
        r = requests.post('https://html.duckduckgo.com/html/',
                          data={'q': query}, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(r.text, 'html.parser')
        urls = []
        for a in soup.select('a.result__a'):
            href = a.get('href')
            if href and href.startswith('http'):
                urls.append(href)
            if len(urls) >= max_results:
                break
        return urls
    except Exception as e:
        print('Search failed:', e)
        return []

query = ' '.join(keywords(sample_doc))
found = search_urls(query)
print('Query:', query)
for u in found:
    print(' -', u)

In [ ]:
# 5. Scrape main text from a page (BeautifulSoup)
def scrape_text(url, max_chars=5000):
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(r.text, 'html.parser')
        for tag in soup(['script', 'style', 'nav', 'header', 'footer']):
            tag.decompose()
        text = ' '.join(p.get_text(' ', strip=True) for p in soup.find_all('p'))
        return re.sub(r'\s+', ' ', text)[:max_chars]
    except Exception as e:
        print('Scrape failed for', url, ':', e)
        return ''

# Fallback URLs so the demo always works even without internet search
urls = found or ['https://en.wikipedia.org/wiki/Photosynthesis']
scraped = {u: scrape_text(u) for u in urls[:10]}
for u, t in scraped.items():
    print(f'{len(t):5d} chars  {u}')

In [ ]:
# 6. Similarity between the document and each scraped page
doc_emb = model.encode(sample_doc, convert_to_tensor=True)
print(f'{"similarity":>10}   source')
results = []
for url, text in scraped.items():
    if not text:
        continue
    page_emb = model.encode(text, convert_to_tensor=True)
    score = float(util.cos_sim(doc_emb, page_emb))
    results.append((score, url))
for score, url in sorted(results, reverse=True):
    flag = '  <-- possible plagiarism' if score >= 0.7 else ''
    print(f'{score:10.3f}   {url}{flag}')

In [ ]:
# 7. Sentence-level: which sentences of the doc match the top source
def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.strip()) > 10]

if results:
    top_url = sorted(results, reverse=True)[0][1]
    doc_sents = split_sentences(sample_doc)
    src_sents = split_sentences(scraped[top_url])
    if src_sents:
        d_emb = model.encode(doc_sents, convert_to_tensor=True)
        s_emb = model.encode(src_sents, convert_to_tensor=True)
        sim = util.cos_sim(d_emb, s_emb)
        print('Matched sentences vs', top_url, '\n')
        for i, sent in enumerate(doc_sents):
            best = float(sim[i].max())
            mark = '[COPIED]' if best >= 0.7 else '[  ok  ]'
            print(f'{mark} {best:.2f}  {sent}')